# Профилирование AlexNet

Запустите ячейки сверху вниз на CUDA. Сначала ноутбук запишет профиль FP32, затем AMP FP16. В каждом трейсе пять шагов, которые записаны после четырёх начальных.

Файлы появятся в `artifacts/traces`. В ячейке с параметрами можно включить сравнение с AutoAugment и без него или проверить разное число workers.

In [1]:
# Пути проекта и импорты
from pathlib import Path
import sys

import torch

ROOT = Path.cwd()
if not (ROOT / "utils" / "alexnet.py").is_file():
    ROOT = ROOT.parent
if not (ROOT / "utils" / "alexnet.py").is_file():
    raise RuntimeError("Откройте ноутбук из корня проекта или из папки notebooks.")
sys.path.insert(0, str(ROOT))

from utils.data_utils import load_training_data

DATA_DIR = ROOT / "data"
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

In [2]:
# Параметры экспериментов и проверка CUDA
SEED = 52
BATCH_SIZE = 512

if not torch.cuda.is_available():
    raise RuntimeError("Для этих экспериментов нужна CUDA. Проверьте выбранное ядро.")
DEVICE = torch.device("cuda")
print(torch.cuda.get_device_name(DEVICE), "PyTorch", torch.__version__)

from utils.profiling import capture, benchmark_workers

WORKERS = 2
RUN_AUGMENT_ABLATION = False
RUN_WORKER_STUDY = False

NVIDIA GeForce RTX 2080 Ti PyTorch 2.14.1+cu130


In [3]:
# Загрузка данных и расчёт статистик train
data = load_training_data(DATA_DIR, SEED)

In [4]:
capture(
    data, DEVICE, ARTIFACTS / "traces" / "training_trace_fp32.json",
    seed=SEED, batch_size=BATCH_SIZE, workers=WORKERS,
)

USDT:2026-11-08 21:17:55 4681:4681 SyncActivityProfilerHandler.cpp:39] profiler_start
USDT:2026-11-08 21:17:56 4681:4681 SyncActivityProfilerHandler.cpp:46] profiler_stop


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                             aten::convolution_backward         0.30%       2.157ms         0.55%       4.029ms     161.161us     229.620ms        32.42%     233.940ms       9.358ms            25  
                                             03_forward         0.00%       0.000us         0.00%       0.000us       0.000us     185.066ms        26.13%     185.066ms      37.013ms             5  
void cudn

In [5]:
capture(
    data, DEVICE, ARTIFACTS / "traces" / "training_trace_amp_fp16.json",
    seed=SEED, batch_size=BATCH_SIZE, workers=WORKERS, use_amp=True,
)

USDT:2026-11-08 21:18:02 4681:4681 SyncActivityProfilerHandler.cpp:39] profiler_start
USDT:2026-11-08 21:18:02 4681:4681 SyncActivityProfilerHandler.cpp:46] profiler_stop


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                             03_forward         0.00%       0.000us         0.00%       0.000us       0.000us     114.157ms        32.20%     114.157ms      22.831ms             5  
                             aten::convolution_backward         0.62%       2.431ms         1.19%       4.654ms     186.170us      85.215ms        24.04%      87.661ms       3.506ms            25  
         

In [6]:
if RUN_AUGMENT_ABLATION:
    for augment, label in ((True, "with"), (False, "without")):
        capture(
            data, DEVICE, ARTIFACTS / "traces" / f"training_trace_{label}_autoaugment.json",
            seed=SEED, batch_size=BATCH_SIZE, workers=0, augment=augment,
        )

In [7]:
if RUN_WORKER_STUDY:
    import pandas as pd

    worker_results = []
    for workers in (0, 2, 4):
        ms = benchmark_workers(data, DEVICE, seed=SEED, batch_size=BATCH_SIZE, workers=workers)
        worker_results.append({"num_workers": workers, "ms_per_batch": ms})
        print(f"{workers} workers | {ms:.2f} мс/батч")

    pd.DataFrame(worker_results).to_csv(ARTIFACTS / "worker_benchmark.csv", index=False)